# Extract clean dfs then references from xlsx files

This is a trial area for the extract, all tests and extraction should be properly carried out in extract_from_xlsx.py, main.py and tests/

In [ ]:
import glob
import os
import re

import numpy as np
import pandas as pd
from pandas.api.types import is_string_dtype

import union_lists.dataset.extract_from_xlsx as extract
import union_lists.dataset.reformat_union_lists as ref

In [ ]:
SCALE = "One Inch"

In [ ]:
xlsx_files = glob.glob(f"../data/raw/{SCALE}/*.xlsx")
xlsx_files = [x for x in xlsx_files if "\\~" not in x and "(2)" not in x]

In [ ]:
dfs = {os.path.basename(x).split(".")[0]: pd.read_excel(x) for x in xlsx_files}

### Exploratory notes

In [ ]:
clean_dfs = {block: extract.pre_process_xlsx(df, block) for block, df in dfs.items()}

In [ ]:
combined_df = pd.concat([df for df in clean_dfs.values()]).reset_index(drop=True)
combined_df.info()

In [ ]:
xlsx_entry_dfs = {}
for file_id, df in clean_dfs.items():
    entries = []
    [entries.extend(ref.process_xlsx_row(row=row[1], source=file_id, scale=SCALE)) for row in df.iterrows()]
    
    entry_df = pd.concat([pd.DataFrame(x, index=[0]) for x in entries]).reset_index(drop=True)
    xlsx_entry_dfs[file_id] = entry_df

In [ ]:
xlsx_combined_df = pd.concat([df for df in xlsx_entry_dfs.values()])

In [ ]:
xlsx_combined_df.columns

#### check clean NA counts

In [ ]:
non_na_count_df = pd.concat([df.count() / len(df) for block, df in clean_dfs.items()], axis=1).T
non_na_count_df = non_na_count_df.where(~non_na_count_df.isna(), 0).astype(float)

axs = non_na_count_df.hist(figsize=(20,15), bins=np.arange(0,1.05,0.025))
[ax.set_xlim(0,1) for ax in axs.flatten()];
[ax.set_ylim(0,40) for ax in axs.flatten()];
[ax.hlines(37, 0, 1, colors='black', linestyle='dashed') for ax in axs.flatten()]
[ax.set_title(col) for col, ax in zip(clean_dfs["Block 2"].columns, axs.flatten())]
axs[0,0].figure;

#### Third row always containes the column headers
- Fourth row the labels for coloured/gridded/copies

In [ ]:
pd.concat([df.iloc[3] for f, df in dfs.items()], axis=1).T

In [ ]:
cols = pd.concat([df.iloc[3] for f, df in dfs.items()], axis=1).T.iloc[3]
cols.values